# Taxonomy and risk-driven scenario generation

This notebook runs the **taxonomy and risk-driven** workflow in [Asago Scenario Generator](https://github.com/asago-ai/asago-scenario-generator). That workflow is a peer of the STPA pipeline (see `stpa-demo.ipynb`), not a legacy mode.

It takes a **use-case description** and a **policy-mapper risk extraction**, maps those risks through NIST / OWASP / MITRE ATLAS via SSSOM, then generates adversarial scenarios, Gherkin behaviour specs, evaluation evidence, and an HTML report.

**What you'll see:**
1. Choose a bundled system (Klarna, OcciAI, or Airbnb) and inspect its inputs
2. Optionally supply a reviewed capability profile and qualification facts
3. Run `run_pipeline` (stages 1–4: profile → threat surface → candidates → scenarios)
4. Inspect the capability profile, threat surface, admitted vs quarantined candidates, scenarios, eval scorecard, and HTML report

---

### Prerequisites

1. **Python virtual environment** — from the repo root:
   ```bash
   uv sync --extra scenario-generator
   ```
   In Jupyter, select this venv as your kernel: **Kernel → Change Kernel** → choose the `.venv` Python interpreter.

2. **A vLLM (or OpenAI-compatible) endpoint** serving `gemma-4-26b-a4b-it`. This is the model used in the live runs this demo is based on.

3. **SSSOM mappings** — bundled at `inputs/mappings/risk_to_category.sssom.tsv` (the same `risk_to_category` file produced by Asago Policy Mapper).

Set these environment variables before launching Jupyter, or edit the config cell:

```bash
export ASAGO_SCENARIO_GENERATOR_MODEL_BASE_URL="https://your-vllm-endpoint.com/v1"
export ASAGO_SCENARIO_GENERATOR_MODEL_NAME="gemma-4-26b-a4b-it"
export ASAGO_SCENARIO_GENERATOR_API_KEY="none"
export ASAGO_SCENARIO_GENERATOR_TEMPERATURE="0.4"
export ASAGO_SCENARIO_GENERATOR_RUN_LIVE="1"
export ASAGO_SCENARIO_GENERATOR_GENERATION_MODE="coverage"
export ASAGO_SCENARIO_GENERATOR_MAX_SCENARIO_TECHNIQUES="1"
export ASAGO_SCENARIO_GENERATOR_MAX_SCENARIOS_PER_PATTERN="1"
```

To use the validated ignored local profile instead of direct endpoint variables, set `ASAGO_MODEL_PROFILE=gemma4-oc-taxonomy` and `ASAGO_MODEL_PROFILES_FILE=/absolute/path/to/config/model-profiles.yaml`. Named profile values take precedence over environment values.

To inspect an existing run instead of calling the model, set `ASAGO_EXISTING_RUN_DIR` to a completed run directory (the child folder that contains `run-manifest.yaml`).


## 1. Configuration

Edit the values below if environment variables are not set.

> **Note:** The notebook defaults to the validated one-ingress **coverage canary**. Set `ASAGO_SCENARIO_GENERATOR_GENERATION_MODE=exhaustive`, `ASAGO_SCENARIO_GENERATOR_PROFILE_SCOPE=full`, `ASAGO_SCENARIO_GENERATOR_MAX_SCENARIO_TECHNIQUES=2`, and leave the per-pattern cap empty for the full corpus. The full Klarna configuration selects about 115 candidates and can take around two hours.


In [ ]:
import os

BASE_URL = os.environ.get("ASAGO_SCENARIO_GENERATOR_MODEL_BASE_URL", "http://localhost:8000/v1")
MODEL = os.environ.get("ASAGO_SCENARIO_GENERATOR_MODEL_NAME", "gemma-4-26b-a4b-it")
API_KEY = os.environ.get("ASAGO_SCENARIO_GENERATOR_API_KEY", "none")
TEMPERATURE = float(os.environ.get("ASAGO_SCENARIO_GENERATOR_TEMPERATURE", "0.4"))
RUN_LIVE = os.environ.get("ASAGO_SCENARIO_GENERATOR_RUN_LIVE", "").lower() in {"1", "true", "yes"}
GENERATION_MODE = os.environ.get("ASAGO_SCENARIO_GENERATOR_GENERATION_MODE", "coverage")
PROFILE_SCOPE = os.environ.get("ASAGO_SCENARIO_GENERATOR_PROFILE_SCOPE", "canary")
MAX_SCENARIO_TECHNIQUES = int(os.environ.get("ASAGO_SCENARIO_GENERATOR_MAX_SCENARIO_TECHNIQUES", "1"))
_pattern_cap = os.environ.get("ASAGO_SCENARIO_GENERATOR_MAX_SCENARIOS_PER_PATTERN", "1").strip()
MAX_SCENARIOS_PER_PATTERN = int(_pattern_cap) if _pattern_cap else None
EXISTING_RUN_DIR = os.environ.get("ASAGO_EXISTING_RUN_DIR", "").strip()
MODEL_PROFILE = os.environ.get("ASAGO_MODEL_PROFILE", "").strip() or None
PROFILES_FILE = os.environ.get("ASAGO_MODEL_PROFILES_FILE", "config/model-profiles.yaml")

for thread_var in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
):
    os.environ.setdefault(thread_var, "1")

os.environ["ASAGO_SCENARIO_GENERATOR_MODEL_BASE_URL"] = BASE_URL
os.environ["ASAGO_SCENARIO_GENERATOR_MODEL_NAME"] = MODEL
os.environ["ASAGO_SCENARIO_GENERATOR_API_KEY"] = API_KEY
os.environ["ASAGO_SCENARIO_GENERATOR_TEMPERATURE"] = str(TEMPERATURE)

print(f"Model:        {MODEL}")
print(f"Temperature:  {TEMPERATURE}")
print(f"Model config: {MODEL_PROFILE or '(environment variables)'}")
print(f"Profiles file:{PROFILES_FILE if MODEL_PROFILE else '(not used)'}")
print(f"Live calls:   {RUN_LIVE}")
print(f"Mode:         {GENERATION_MODE}")
print(f"Profile scope:{PROFILE_SCOPE}")
print(f"Max techniques per scenario: {MAX_SCENARIO_TECHNIQUES}")
print(f"Max scenarios per pattern:   {MAX_SCENARIOS_PER_PATTERN}")
print(f"Existing run: {EXISTING_RUN_DIR or '(none — will generate)'}")


In [ ]:
from pathlib import Path
import os
import json

import ipywidgets as widgets
import pandas as pd
import yaml
from IPython.display import display, HTML, Markdown


def find_inputs_dir() -> Path:
    here = Path.cwd().resolve()
    candidates = [
        here / "inputs",
        here / "asago-scenario-generator" / "inputs",
        here.parent / "asago-scenario-generator" / "inputs",
    ]
    for candidate in candidates:
        if candidate.is_dir() and (candidate / "use-cases").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find asago-scenario-generator/inputs. "
        "Run Jupyter from the repo root or the asago-scenario-generator folder."
    )


INPUTS = find_inputs_dir()
OUTPUT_ROOT = INPUTS.parent / "output"
OUTPUT_ROOT.mkdir(exist_ok=True)

SYSTEMS = {
    "Klarna (fintech CS agent)": {
        "use_case": INPUTS / "use-cases" / "use-case-klarna-fs-isac-v36.txt",
        "risk_extraction": INPUTS / "risk-extractions" / "risk-extraction-fs-isac.json",
        "profile": INPUTS / "profiles" / "klarna-capability-profile.yaml",
        "canary_profile": INPUTS / "profiles" / "klarna-direct-canary-profile.yaml",
        "qualification_facts": INPUTS / "profiles" / "klarna-qualification-facts.yaml",
        "policy": "FS-ISAC generative AI policy",
    },
    "OcciAI (NHS patient portal)": {
        "use_case": INPUTS / "use-cases" / "use-case-occiAI-guy-nhs-v1.txt",
        "risk_extraction": INPUTS / "risk-extractions" / "risk-extraction-guy-nhs.json",
        "profile": None,
        "canary_profile": None,
        "qualification_facts": None,
        "policy": "Guy's and St Thomas' NHS AI policy",
    },
    "Airbnb (travel support platform)": {
        "use_case": INPUTS / "use-cases" / "uc-airbnb-conversational-ai-platform.md",
        "risk_extraction": INPUTS / "risk-extractions" / "risk-extraction-amadeus.json",
        "profile": None,
        "canary_profile": None,
        "qualification_facts": None,
        "policy": "Amadeus responsible AI policy",
    },
}

SSSOM_PATH = INPUTS / "mappings" / "risk_to_category.sssom.tsv"

print(f"Inputs:  {INPUTS}")
print(f"Output:  {OUTPUT_ROOT}")


## 2. Select a system

Each option pairs a use-case description with the policy risk extraction used in prior live runs:

| System | Use case | Risk extraction |
|--------|----------|-----------------|
| Klarna | Consumer fintech CS agent | FS-ISAC |
| OcciAI | NHS outpatient patient portal | Guy's and St Thomas' NHS |
| Airbnb | Hybrid travel support platform | Amadeus |


In [ ]:
system_dropdown = widgets.Dropdown(
    options=list(SYSTEMS.keys()),
    value="Klarna (fintech CS agent)",
    description="System:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="520px"),
)
use_profile_checkbox = widgets.Checkbox(
    value=True,
    description="Use reviewed capability profile + qualification facts when available",
    indent=False,
)
display(system_dropdown, use_profile_checkbox)


def selected_system() -> dict:
    return SYSTEMS[system_dropdown.value]


## 3. Preview inputs

Risk extractions are filtered to IBM Risk Atlas cards — that is the contract `run_pipeline` consumes.


In [ ]:
from asago_scenario_generator.data.loaders import load_risk_extraction

sys = selected_system()
use_case_text = sys["use_case"].read_text(encoding="utf-8")
risk_cards = load_risk_extraction(sys["risk_extraction"])

print(f"System:           {system_dropdown.value}")
print(f"Policy source:    {sys['policy']}")
print(f"Use case:         {sys['use_case'].name} ({len(use_case_text):,} chars)")
print(f"Risk extraction:  {sys['risk_extraction'].name}")
print(f"Atlas risk cards: {len(risk_cards)}")
print(f"SSSOM mappings:   {SSSOM_PATH.name}")
print()
print("Use-case excerpt:")
print("-" * 60)
print("\n".join(use_case_text.strip().splitlines()[:18]))
print("...")

risks_df = pd.DataFrame(
    [
        {
            "Risk ID": c.risk_id,
            "Name": c.risk_name,
            "Confidence": round(c.confidence, 3) if c.confidence is not None else None,
            "Grounding": c.grounding_confidence,
            "Evidence": len(c.evidence or []),
        }
        for c in risk_cards
    ]
)
print()
print(f"Showing {min(15, len(risks_df))} of {len(risks_df)} IBM Risk Atlas cards:")
risks_df.head(15)


## 4. Run the taxonomy/risk pipeline

This calls `asago_scenario_generator.pipeline.runner.run_pipeline`, the same entry point as:

```bash
uv run asago-scenario-generator generate \
  --use-case @use-case.txt \
  --risk-extraction risk-extraction.json \
  --sssom mappings.sssom.tsv \
  --output-dir output/taxonomy-risk \
  --profile capability-profile.yaml \
  --qualification-facts facts.yaml \
  --generation-mode coverage \
  --max-scenario-techniques 1 \
  --max-scenarios-per-pattern 1
```

The default coverage settings are a bounded live release gate. Live calls require `ASAGO_SCENARIO_GENERATOR_RUN_LIVE=1`; without it, the notebook only inspects `ASAGO_EXISTING_RUN_DIR`. Use the full profile and exhaustive settings only when you intentionally want the complete corpus. Do not treat process exit or legacy manifest counters alone as success: inspect `finalization-inventory.json` and verify one YAML/feature pair per admission.


In [ ]:
import inspect

from asago_scenario_generator.pipeline.runner import run_pipeline

required_run_parameters = {"generation_mode", "max_techniques", "max_scenarios_per_pattern", "qualification_facts_path", "model_profile", "profiles_file"}
missing_run_parameters = required_run_parameters - set(inspect.signature(run_pipeline).parameters)
if missing_run_parameters and not EXISTING_RUN_DIR:
    raise RuntimeError(
        "The installed asago-scenario-generator is too old for this example; "
        f"run_pipeline is missing: {sorted(missing_run_parameters)}. "
        "Install a revision that includes exhaustive generation and qualification facts."
    )

sys = selected_system()
use_case_text = sys["use_case"].read_text(encoding="utf-8")
collection_dir = OUTPUT_ROOT / "taxonomy-risk"
collection_dir.mkdir(parents=True, exist_ok=True)

profile_key = "canary_profile" if PROFILE_SCOPE == "canary" else "profile"
profile_path = sys.get(profile_key) if use_profile_checkbox.value else None
facts_path = sys["qualification_facts"] if use_profile_checkbox.value else None
if profile_path is None:
    print(
        "No reviewed capability profile for this system. "
        "Stage 1 will infer a profile; the run may admit zero scenarios "
        "if trust_boundaries / external_integrations / qualification facts are missing."
    )

if EXISTING_RUN_DIR:
    run_dir = Path(EXISTING_RUN_DIR)
    if not (run_dir / "run-manifest.yaml").exists():
        raise FileNotFoundError(f"No run-manifest.yaml in {run_dir}")
    print(f"Skipping live generation; inspecting {run_dir}")
    result = None
else:
    if not RUN_LIVE:
        raise RuntimeError(
            "Live model calls are disabled. Set ASAGO_SCENARIO_GENERATOR_RUN_LIVE=1 "
            "or set ASAGO_EXISTING_RUN_DIR to inspect an existing run."
        )
    if MODEL_PROFILE and not Path(PROFILES_FILE).is_file():
        raise FileNotFoundError(f"Model profiles file not found: {PROFILES_FILE}")
    model_kwargs = (
        {"model_profile": MODEL_PROFILE, "profiles_file": Path(PROFILES_FILE)}
        if MODEL_PROFILE
        else {"base_url": BASE_URL, "api_key": API_KEY, "model": MODEL}
    )
    print(f"Generating scenarios for: {system_dropdown.value}")
    print(f"Collection dir: {collection_dir}")
    result = run_pipeline(
        use_case=use_case_text,
        risk_extraction_path=sys["risk_extraction"],
        sssom_path=SSSOM_PATH,
        output_dir=collection_dir,
        profile_path=profile_path,
        qualification_facts_path=facts_path,
        max_techniques=MAX_SCENARIO_TECHNIQUES,
        max_scenarios_per_pattern=MAX_SCENARIOS_PER_PATTERN,
        generation_mode=GENERATION_MODE,
        eval=True,
        log_level="INFO",
        **model_kwargs,
    )
    run_dir = Path(result.run_dir)
    print()
    print("Pipeline complete.")
    print(f"  Manifest status:       {result.manifest_status.value}")
    print(f"  Candidates admitted:   {result.admitted_count}")
    print(f"  Candidates quarantined:{result.quarantined_count}")
    print(f"  Candidates failed:     {result.failed_count}")
    print(f"  Scenario objects:       {len(result.scenarios)}")
    print(f"  Governance-only:       {result.governance_only_count}")
    print(f"  Run directory:         {run_dir}")


## 5. Run manifest and authoritative admission inventory


In [ ]:
manifest_path = run_dir / "run-manifest.yaml"
manifest = yaml.safe_load(manifest_path.read_text(encoding="utf-8"))

print(f"Run ID:     {manifest.get('run_id')}")
print(f"Status:     {manifest.get('status')}")
print(f"Started:    {manifest.get('timestamp_start')}")
print(f"Ended:      {manifest.get('timestamp_end')}")
print()

finalization_path = run_dir / "finalization-inventory.json"
if finalization_path.exists():
    finalization = json.loads(finalization_path.read_text(encoding="utf-8"))
    decisions = finalization.get("admission_decisions") or []
    admitted_from_inventory = sum(d.get("admitted") is True for d in decisions)
    quarantined_from_inventory = len(decisions) - admitted_from_inventory
    print("Authoritative finalization inventory:")
    print(f"  Attempted:   {len(decisions)}")
    print(f"  Admitted:    {admitted_from_inventory}")
    print(f"  Quarantined: {quarantined_from_inventory}")
    if decisions:
        print(f"  Admission:   {admitted_from_inventory / len(decisions):.1%}")
    legacy_generated = manifest.get("scenarios_generated")
    if legacy_generated not in (None, admitted_from_inventory):
        print(
            f"WARNING: legacy manifest scenarios_generated={legacy_generated} "
            f"disagrees with finalization inventory={admitted_from_inventory}."
        )
    print()
else:
    finalization = {}
    decisions = []
    print("No finalization-inventory.json; this run is not fully inspectable.")
    print()

inventory = manifest.get("inventory") or []
roles = {}
for entry in inventory:
    roles.setdefault(entry.get("role"), 0)
    roles[entry["role"]] += 1
print("Inventory roles:")
for role, count in sorted(roles.items()):
    print(f"  {role}: {count}")

if result is not None:
    print()
    if result.generation_notes:
        print("Generation notes:")
        for note in result.generation_notes:
            print(f"  - {note}")


## 6. Capability profile

The profile is either inferred (Stage 1) or taken from the reviewed YAML. Authoritative projection uses zones, entry points, tools, trust boundaries, and external integrations.


In [ ]:
if result is not None:
    profile = result.capability_profile
    profile_data = profile.model_dump(mode="json")
else:
    profile_data = yaml.safe_load((run_dir / "capability-profile.yaml").read_text(encoding="utf-8"))

zones = profile_data.get("zones_active") or []
entry_points = profile_data.get("entry_points") or []
tools = profile_data.get("tool_inventory") or []
boundaries = profile_data.get("trust_boundaries") or []
integrations = profile_data.get("external_integrations") or []

print(f"Confidence:     {profile_data.get('confidence')}")
print(f"Active zones:   {', '.join(zones) or '(none)'}")
print(f"KC subcodes:    {', '.join(profile_data.get('kc_subcodes') or [])}")
print(f"Entry points:   {len(entry_points)}")
print(f"Tools:          {len(tools)}")
print(f"Trust bounds:   {len(boundaries)}")
print(f"Integrations:   {len(integrations)}")

if entry_points:
    display(pd.DataFrame(
        [
            {
                "Name": ep.get("name") if isinstance(ep, dict) else getattr(ep, "name", ep),
                "Direction": ep.get("direction") if isinstance(ep, dict) else getattr(ep, "direction", ""),
                "Controllability": ep.get("controllability") if isinstance(ep, dict) else getattr(ep, "controllability", ""),
            }
            for ep in entry_points
        ]
    ))


## 7. Threat surface

Deterministic three-hop mapping: Risk Atlas → OWASP LLM Top 10 (SSSOM) → OWASP agentic threats → ATLAS techniques, gated by the capability profile.


In [ ]:
if result is not None:
    surface = result.threat_surface
    entries = [e.model_dump(mode="json") for e in surface.entries]
    gov_only = [e.model_dump(mode="json") for e in surface.governance_only]
else:
    surface_data = yaml.safe_load((run_dir / "threat-surface.yaml").read_text(encoding="utf-8"))
    entries = surface_data.get("entries") or []
    gov_only = surface_data.get("governance_only") or []

print(f"In-scope entries: {len(entries)}")
print(f"Governance-only:  {len(gov_only)}")

rows = []
for entry in entries:
    card = entry.get("risk_card") or {}
    rows.append({
        "Risk": card.get("risk_id") or card.get("id") or "",
        "OWASP LLM": ", ".join(entry.get("owasp_llm_ids") or []),
        "Agentic T": ", ".join(entry.get("agentic_threat_ids") or []),
        "ATLAS": ", ".join(entry.get("atlas_technique_ids") or []),
        "Patterns": ", ".join(entry.get("attack_pattern_ids") or []),
        "Gov-only": entry.get("governance_only", False),
    })
surface_df = pd.DataFrame(rows)
surface_df.head(25)


## 8. Generated scenarios

Admitted scenarios are written under the run directory. Quarantined candidates stay in `quarantine/` with the reason they were not admitted — that is expected, not a crash.


In [ ]:
from IPython.display import display, HTML

scenario_files = sorted(run_dir.glob("scenarios/**/*.yaml")) + sorted(run_dir.glob("*.scenario.yaml"))
quarantine_files = sorted((run_dir / "quarantine").glob("*.json")) if (run_dir / "quarantine").exists() else []

live_scenarios = list(result.scenarios) if result is not None else []
print(f"Admitted scenarios (result object): {len(live_scenarios)}")
print(f"Scenario YAML files:                {len(scenario_files)}")
print(f"Quarantine bundles:                 {len(quarantine_files)}")

scenario_rows = []
scenario_objs = []
if live_scenarios:
    for scn in live_scenarios:
        scenario_objs.append(scn)
        scenario_rows.append({
            "Scenario ID": scn.scenario_id,
            "Title": scn.narrative.title,
            "Entry point": scn.narrative.entry_point,
            "Zones": " → ".join(scn.narrative.zone_sequence),
            "Priority": round(scn.priority.composite, 3) if scn.priority else None,
            "Risk": scn.faceting.risk_card.risk_id if scn.faceting and scn.faceting.risk_card else "",
        })
else:
    for path in scenario_files:
        data = yaml.safe_load(path.read_text(encoding="utf-8"))
        narrative = data.get("narrative") or {}
        faceting = data.get("faceting") or {}
        risk = (faceting.get("risk_card") or {}).get("risk_id", "")
        scenario_objs.append(data)
        scenario_rows.append({
            "Scenario ID": data.get("scenario_id", path.stem),
            "Title": narrative.get("title", path.name),
            "Entry point": narrative.get("entry_point", ""),
            "Zones": " → ".join(narrative.get("zone_sequence") or []),
            "Priority": (data.get("priority") or {}).get("composite"),
            "Risk": risk,
        })

if scenario_rows:
    display(pd.DataFrame(scenario_rows))
else:
    print("No admitted scenarios in this run. Inspect quarantine and coverage-gaps.json next.")


In [ ]:
def _as_dict(obj):
    if hasattr(obj, "model_dump"):
        return obj.model_dump(mode="json")
    return obj


def render_taxonomy_scenario(obj, index: int) -> str:
    data = _as_dict(obj)
    narrative = data.get("narrative") or {}
    steps = narrative.get("steps") or []
    faceting = data.get("faceting") or {}
    chain = faceting.get("taxonomy_chain") or {}
    priority = data.get("priority") or {}
    html = '<div style="border:1px solid #ddd; padding:16px; margin:8px 0; border-radius:8px; background:#fafafa;">'
    html += f'<h3 style="margin-top:0;">#{index + 1} {narrative.get("title") or data.get("scenario_id")}</h3>'
    html += f'<p><b>ID:</b> <code>{data.get("scenario_id")}</code></p>'
    if narrative.get("summary"):
        html += f'<p>{narrative["summary"]}</p>'
    html += (
        f'<p><b>Entry:</b> {narrative.get("entry_point", "")} &nbsp; '
        f'<b>Zones:</b> {" → ".join(narrative.get("zone_sequence") or [])} &nbsp; '
        f'<b>Priority:</b> {priority.get("composite", "")}</p>'
    )
    if chain:
        risk_id = (faceting.get("risk_card") or {}).get("risk_id", "")
        html += (
            "<p><b>Taxonomy chain:</b> "
            f"{risk_id} → "
            f'{", ".join(chain.get("owasp_llm_ids") or [])} → '
            f'{", ".join(chain.get("agentic_threat_ids") or [])} → '
            f'{", ".join(chain.get("atlas_technique_ids") or [])}</p>'
        )
    if steps:
        html += "<h4>Narrative steps</h4><ol>"
        for step in steps:
            zone = step.get("zone", "")
            action = step.get("action") or ""
            effect = step.get("effect") or ""
            html += f'<li><span style="color:#666;">{zone}</span> {action}'
            if effect:
                html += f' → <em>{effect}</em>'
            html += "</li>"
        html += "</ol>"
    html += "</div>"
    return html


if scenario_objs:
    for i, obj in enumerate(scenario_objs[:8]):
        display(HTML(render_taxonomy_scenario(obj, i)))
    if len(scenario_objs) > 8:
        print(f"... and {len(scenario_objs) - 8} more scenarios in {run_dir}")


## 9. Quarantine and coverage

Candidates that fail admission gates are preserved with evidence. Coverage artefacts record which attack patterns were planned vs produced.


In [ ]:
coverage_plan = run_dir / "coverage-plan.json"
coverage_gaps = run_dir / "coverage-gaps.json"
if coverage_plan.exists():
    plan = json.loads(coverage_plan.read_text(encoding="utf-8"))
    print("Coverage plan keys:", ", ".join(sorted(plan.keys())[:20]))
if coverage_gaps.exists():
    gaps = json.loads(coverage_gaps.read_text(encoding="utf-8"))
    print("Coverage gaps keys:", ", ".join(sorted(gaps.keys())[:20]))
    print()

print(f"Quarantine bundles: {len(quarantine_files)}")
qrows = []
for path in quarantine_files:
    data = json.loads(path.read_text(encoding="utf-8"))
    candidate = data.get("candidate") or data
    violations = data.get("violations") or []
    if not violations:
        violations = [{"code": "unknown", "owner": None, "detail": data.get("reason", "")}]
    for violation in violations:
        qrows.append({
            "Candidate": candidate.get("candidate_id") or data.get("candidate_id") or path.stem,
            "Owner": violation.get("owner") or "application",
            "Code": violation.get("code") or "unknown",
            "Detail": (violation.get("detail") or violation.get("message") or "")[:160],
        })
if qrows:
    quarantine_df = pd.DataFrame(qrows)
    display(
        quarantine_df.groupby(["Owner", "Code"], dropna=False)
        .size()
        .rename("Count")
        .reset_index()
        .sort_values("Count", ascending=False)
    )
    display(quarantine_df.head(30))


## 10. Eval scorecard and HTML report

The report is written inside the immutable run directory. Open the HTML file in a browser.


In [ ]:
scorecard_path = run_dir / "eval-scorecard.yaml"
if scorecard_path.exists():
    scorecard = yaml.safe_load(scorecard_path.read_text(encoding="utf-8"))
    print(f"Scenario count: {scorecard.get('scenario_count')}")
    print(f"Feature files:  {scorecard.get('feature_file_count')}")
    presence = (scorecard.get("presence_coverage") or {}).get("metrics") or {}
    if presence:
        print()
        print("Presence coverage:")
        for name, metric in list(presence.items())[:12]:
            status = metric.get("status") if isinstance(metric, dict) else metric
            print(f"  {name}: {status}")

report_path = run_dir / "report.html"
print()
if report_path.exists():
    print(f"HTML report: {report_path}")
    print("Open that file in a browser to review the full run.")
else:
    print("No report.html in this run directory.")
print(f"Run directory: {run_dir}")
